# Material de estudio personal: Annealing Parte 1 - BQM

En este notebook simulare un notebook educativo que hable sobre las bases para poder entender el Simulated Annealing y el Quantum Annealing. Para entender el funcionamiento de este notebook, es necesario estar familiarizado con los siguientes conceptos:

- Introducción a la computación cuántica básica
    - Bits, Qubits, Compuertas, Algoritmos básicos y protocolos básicos
- Introducción a la Mecánica Cuántica Básica
    - Estados cuánticos, superposición, Hamitonianos, Operador de evolución temporal
- Problemas de optimización combinatoria
    - Formulación QUBO para problemas combinatorios
    - Transformación QUBO-Ising

Mucho de este contenido previo lo puedes consultar en mi repositorio CC2-2026-2-UNAM y para la introducción a la computación cuántica básica, en el futuro repositorio CC1-2027-1-UNAM

Buena parte de este contenido esta inspirado en el curso QCobalt del OQI y se tomará como referencia principal, mas no única de estos temas

In [17]:
# Corre esta celda para instalar OceanSDK, de Dwave, si estas trabajando en google collab

!pip install dwave-ocean-sdk

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


 ## Recordatorio: Representaciones QUBO e Ising

Un problema QUBO es un problema de minimización que se puede definir con una función de la siguiente manera:

$$f(x) = \sum\limits_i {Q_{i, i} x_i} + \sum\limits_{i < j} {Q_{i, j} x_i x_j} $$

donde x_i puede tomar valores entre 0 y 1

y un Hamiltoniano con variables tipo Ising

$$f(s)=\sum\limits_{i} h_i s_i + \sum\limits_{i<j} J_{i,j} s_i s_j 
$$

El `BinaryQuadraticModel` de OceanSDK permite que nosotros podamos representar a una función QUBO o Ising de manera rapida y directa, con la ayuda de diccionarios, para apoyarnos en esto como herramienta para usar con el hardware de D-Wave y las técnicas de Annealing que veremos mas adelante. Con esto en mente, veamos como podemos utilizarla y representar variables QUBO

Los modelos Ising y QUBO tienen en comun 2 cosas:

> Tienen terminos lineales

> Tienen terminos cuadráticos

Y con BQM podemos representarlos a los 2

Vamos a tomar una formulación y vamos a verla en QUBO y como se sustituye en BQM

$$-3x_1-5x_2-7x_3-4x_4+3x_1x_2+4x_2x_3+5x_3x_4+2x_4x_1$$

Lo que debemos hacer es dividir el problema en términos lineales y cuadráticos, y tras ello, colocar cada un en un diccionario que luego el programa va a utilizar. Veamoslo paso a paso:

- Los terminos lineales son: $-3x_1-5x_2-7x_3-4x_4$. A cada uno de los valores de la variable, le asignamos con el diccionario el valor del coeficiente.

In [18]:
linear = {"x1": -3, "x2": -5, "x3": -7, "x4": -4}

- Los términos cuadráticos son $3x_1x_2+4x_2x_3+5x_3x_4+2x_4x_1$. A cada uno de los valores conjuntos de la variable, le asignamos en otro diccionario el valor del coeficiente.

In [19]:
quadratic = {("x1","x2"): 3,("x2","x3"): 4, ("x3","x4"): 5, ("x1","x4"): 2}

Ahora, dependiendo del tipo de variable que quieras usar en tu sistema, el parámetro "Vartype" de BQM puede tomar 2 posibles valores: SPIN o BINARY. 
- Si estas tratando con variables QUBO (entre 0 y 1), se le asigna BINARY
- Si estas tratando con variables ISING (entre -1 y 1), se le asigna SPIN


In [20]:
vartype = "BINARY"  #Si quieres probar con algo diferente, puedes llamarla SPIN

In [21]:
from dimod import BQM

modelo_bqm1 = BQM(linear, quadratic, vartype)

In [22]:
modelo_bqm1

BinaryQuadraticModel({'x1': -3.0, 'x2': -5.0, 'x3': -7.0, 'x4': -4.0}, {('x2', 'x1'): 3.0, ('x3', 'x2'): 4.0, ('x4', 'x1'): 2.0, ('x4', 'x3'): 5.0}, 0.0, 'BINARY')

Si la función tuviera una constante, tambien podemos añadir dicho valor como un OFFSET que nos dice basicamente, la función se desplaza cierto valor. Por ejemplo, si la función anterior fuera:

$$-3x_1-5x_2-7x_3-4x_4+3x_1x_2+4x_2x_3+5x_3x_4+2x_4x_1+5$$

In [23]:
offset = 5

In [24]:
from dimod import BQM

modelo_bqm1_offset = BQM(linear, quadratic, offset, vartype)

In [25]:
modelo_bqm1_offset

BinaryQuadraticModel({'x1': -3.0, 'x2': -5.0, 'x3': -7.0, 'x4': -4.0}, {('x2', 'x1'): 3.0, ('x3', 'x2'): 4.0, ('x4', 'x1'): 2.0, ('x4', 'x3'): 5.0}, 5.0, 'BINARY')

Puedes probar [Tu nombre del modelo].offset, .vartype, .linear o .quadratic para poder visualizar los parámetros. Ejemplo:

In [26]:
modelo_bqm1.vartype

<Vartype.BINARY: frozenset({0, 1})>

## Encontrando la solución optima de manera clásica

La filosofía de trabajo de OceanSDK es muy simple. Una vez tienes algo representado en BQM, `OceanSDK` se encarga de generar solucionadores convenientes para las herramientas con las que tu vayas a trabajar. Y precisamente diseña herramientas para poder trabajar con los distintos tipos de Annealing (sea lo que sea que eso signifique). Pero antes de entrar en ellos, entenderemos como `OceanSDK` provee herramientas para resolver este asunto

Existen solucionadores clásicos que permiten encontrar la solución al sistema de manera rápida. Sin embargo, no son eficientes, pues utilizan fuerza bruta para poder encontrar la solución óptima. 

Sin embargo, vale la pena verlos antes de entrar al mundo del Annealing y empezar a obsesionarnos con los métodos de solución avanzados, para entender el alcance de las herramientas que tenemos a la mano.

El sampler `ExactSolver()` precisamente tiene la intención de resolverlo por medio de fuerza bruta, al procesar todos los valores posibles y acomodar las energías a modo de top.

In [27]:
from dimod import BinaryQuadraticModel
from dimod.reference.samplers import ExactSolver

#Usaremos el modelo de BQM que creamos antes, para resolverlo con ExactSolver

sampler = ExactSolver() #El sampler que resuelve BQM
sampleset = sampler.sample(modelo_bqm1) # Colocamos el modelo a resolver

In [28]:
print(sampleset)

   x1 x2 x3 x4 energy num_oc.
6   1  0  1  0  -10.0       1
12  0  1  0  1   -9.0       1
4   0  1  1  0   -8.0       1
5   1  1  1  0   -8.0       1
7   0  0  1  0   -7.0       1
9   1  0  1  1   -7.0       1
11  0  1  1  1   -7.0       1
13  1  1  0  1   -7.0       1
8   0  0  1  1   -6.0       1
2   1  1  0  0   -5.0       1
3   0  1  0  0   -5.0       1
10  1  1  1  1   -5.0       1
14  1  0  0  1   -5.0       1
15  0  0  0  1   -4.0       1
1   1  0  0  0   -3.0       1
0   0  0  0  0    0.0       1
['BINARY', 16 rows, 16 samples, 4 variables]


El Sampleset que generamos es el conjunto de soluciones de energía que nuestro BQM tiene a su disposición. Con esto en mente, veamos que podemos hacer con el Sampleset:

- `.first`: Obtiene la mejor solución

In [29]:
print(sampleset.first)

Sample(sample={'x1': np.int8(1), 'x2': np.int8(0), 'x3': np.int8(1), 'x4': np.int8(0)}, energy=np.float64(-10.0), num_occurrences=np.int64(1))


- `.first.sample`: Obtiene unicamente los parámetros solución

In [30]:
print(sampleset.first.sample)

{'x1': np.int8(1), 'x2': np.int8(0), 'x3': np.int8(1), 'x4': np.int8(0)}


- `first.energy`: Nos da la energía mínima del sistema, nada mas

In [32]:
print(sampleset.first.energy)

-10.0


- `.lowest()`: Si existen multiples valores mínimos de energía, entonces la función lowest permite encontrar los múltiples mínimos de energía de la siguiente manera

Para eso, revisemos un ejemplo simple basado en el contenido de QCobalt del OQI

$$ f(x)=x_1x_2$$

In [37]:
from dimod import BQM
from dimod.reference.samplers import ExactSolver

quadratic = {('x1','x2'): 1}
vartype = 'BINARY'

bqm = BQM(quadratic, vartype)

In [38]:
sampleset = sampler.sample(bqm)
print(sampleset)

  x1 x2 energy num_oc.
0  0  0    0.0       1
1  1  0    0.0       1
3  0  1    0.0       1
2  1  1    1.0       1
['BINARY', 4 rows, 4 samples, 2 variables]


In [ ]:
print(sampleset.lowest())

  x1 x2 energy num_oc.
0  0  0    0.0       1
1  1  0    0.0       1
2  0  1    0.0       1
['BINARY', 3 rows, 3 samples, 2 variables]


## Referencias

### Material base

* QWorld. *QCobalt*. QWorld.
  https://qworld.net/qcobalt/

* QWorld & Open Quantum Institute. *QWorld OQI Pre-Hackathon Training Programme*, módulo QCobalt.
  https://qworld.net/qworld-oqi-hackathon-courses/

### Documentación de D-Wave / Ocean SDK

* D-Wave Quantum Inc. *Ocean SDK Documentation: dimod*.
  https://docs.dwavequantum.com/en/latest/ocean/api_ref_dimod/

* D-Wave Quantum Inc. *Binary Quadratic Model*.
  https://docs.dwavequantum.com/en/latest/concepts/models.html#binary-quadratic-model

* D-Wave Quantum Inc. *dimod.binary.BinaryQuadraticModel*.
  https://docs.dwavequantum.com/en/latest/ocean/api_ref_dimod/models.html#binary-quadratic-models

* D-Wave Quantum Inc. *Samplers and Composites: ExactSolver*.
  https://docs.dwavequantum.com/en/latest/ocean/api_ref_dimod/sampler_composites.html#exact-solver

* D-Wave Quantum Inc. *dimod.reference.samplers.ExactSolver.sample*.
  https://docs.dwavequantum.com/en/latest/ocean/api_ref_dimod/generated/dimod.reference.samplers.ExactSolver.sample.html

* D-Wave Quantum Inc. *Samplesets and Solutions*.
  https://docs.dwavequantum.com/en/latest/concepts/samplesets.html

### Referencias teóricas complementarias

* Glover, F., Kochenberger, G., & Du, Y. *A Tutorial on Formulating and Using QUBO Models*. arXiv:1811.11538, 2018.
  https://arxiv.org/abs/1811.11538

* Lucas, A. *Ising formulations of many NP problems*. Frontiers in Physics, 2, 5, 2014.
  https://doi.org/10.3389/fphy.2014.00005
